In [7]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

# Load data
df = pd.read_csv("output.csv")

# Split features + target
X = df.drop(columns=["has_foot_ulcer"])
y = df["has_foot_ulcer"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale (IMPORTANT for neural network)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Train model
model = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300)
model.fit(X_train, y_train)


# Save everything
joblib.dump(model, "dfu_model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(X.columns.tolist(), "feature_order.pkl")

['feature_order.pkl']

In [16]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve
)

# Predictions
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]  # Probability of class 1 (DFU)

# Accuracy (keep this)
print(f"The model is {model.score(X_test, y_test) * 100:.0f}% accurate based off of the dataset and the has_foot_ulcer feature.")

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)

# Classification Report (Precision, Recall, F1)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


The model is 96% accurate based off of the dataset and the has_foot_ulcer feature.

Confusion Matrix:
[[19582    77]
 [  689     6]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.98     19659
           1       0.07      0.01      0.02       695

    accuracy                           0.96     20354
   macro avg       0.52      0.50      0.50     20354
weighted avg       0.94      0.96      0.95     20354



In [9]:
from sklearn.inspection import permutation_importance
import pandas as pd

# Compute feature importance
result = permutation_importance(
    model,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

# Create DataFrame
importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": result.importances_mean
})

# Normalize to percentage
importance_df["Importance (%)"] = (
    importance_df["Importance"] / importance_df["Importance"].sum()
) * 100

# Sort and get top 5
top_features = (importance_df.sort_values(by="Importance", ascending=False).head(5).drop(columns=["Importance"]))

print("\nThese are the 5 most important features when it comes to predicting a diabetic foot ulcer based off of the dataset:\n")
print(top_features)


These are the 5 most important features when it comes to predicting a diabetic foot ulcer based off of the dataset:

            Feature  Importance (%)
14          insulin       33.916555
11      diabetesMed       29.232840
13        metformin       14.885599
12           change       10.659489
5   num_medications        4.279946
